# CS229 Simplified PS3 — Unsupervised Learning

参考 CS229 K-means、PCA 与 semi-supervised EM 作业题型。

这不是 Stanford 官方作业副本，而是用于自学的**简化原创版本**。题型参考公开的
CS229 problem sets；公式以 2026 Main Notes 为准，数据由本仓库确定性生成。

- [Spring 2026 课程主页](https://cs229.stanford.edu/index.html-spr26)
- [Spring 2026 公开视频](https://www.youtube.com/playlist?list=PLaqpC4kq8Gpw)
- [CS229 2026 Main Notes](https://cs229.stanford.edu/main_notes.pdf)
- [Stanford 官方公开的 Summer 2020 problem sets](https://cs229.stanford.edu/summer2020/)
- [公开可见的 Summer 2025 作业结构参考](https://github.com/MDzimah/Stanford-University-CS229-Machine-Learning)

**使用规则：** 先手推，再写代码。不要先看学生 solution。每道题完成后才把对应的
`RUN_*_CHECKS` 改成 `True`。检查不通过时，从 shape 和公式开始定位。

**怎么读题：** `m` 表示样本数，`d` 表示设计矩阵的列数（含截距列时也计入）。
`X` 的每一行是一条样本，`y[i]` 是对应目标；`(m,)` 是一维数组，和 `(m, 1)` 不同。
“实现”指填写函数中的 TODO；“运行检查”指执行已经给出的调用代码；“解释”写在答题框中。
函数定义格和检查格需要分别运行。检查通过只代表已检查的条件成立，不代替解释实验结果。

## Goal

对应 Spring 2026 Lecture 9–10。通过三个可见实验理解：K-means 如何压缩颜色、
PCA 如何寻找主要方向、EM 如何在隐变量存在时交替更新。

## Setup

数据和小图像都由构建脚本生成，不需要联网。

P1 使用 `image`：shape 为 `(48,48,3)`，最后一维依次是 RGB 三个颜色通道，
数值位于 `[0,1]`。P2 使用 `points`：180 个二维点，shape 为 `(180,2)`。
P3 使用该题单独生成的一维 `x_gmm`。三题的数据不同，不需要把前一题的输出作为后一题输入。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(229)

candidates = [Path("data"), Path("problem_sets/ps3_unsupervised/data")]
DATA_DIR = next((path for path in candidates if path.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run: python scripts/build_problem_sets.py")

print("Data directory:", DATA_DIR.resolve())


### 加载并查看本题集的数据

上格准备了库和数据目录。下格读取数据、打印 shape，并展示实验输入；按顺序运行即可。

In [ ]:
points = np.loadtxt(DATA_DIR / "points.csv", delimiter=",", skiprows=1)
image = np.load(DATA_DIR / "tiny_image.npy")
print("points:", points.shape, "image:", image.shape)

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].scatter(points[:, 0], points[:, 1], s=12)
axes[0].set_title("Unlabelled points")
axes[1].imshow(image)
axes[1].set_title("Original tiny image")
axes[1].axis("off")
plt.show()


## Problem 1 — K-means 与图像压缩

**本题目的：** 把图片的许多颜色替换成 4 种代表颜色。先实现分组、更新中心，再把两步循环起来，最后还原成图片。

### 1. assign_clusters

**要写什么：** `assign_clusters(X,centroids)` 为每个样本选最近的中心。
`X:(m,d)`、中心 `centroids:(k,d)`，返回整数编号数组 `assignment:(m,)`。
算平方欧氏距离，取距离最小的中心编号；相同时取编号较小者。
例如一维样本 `[[0.],[9.]]`、中心 `[[1.],[8.]]`，应返回 `[0,1]`。

In [ ]:
def assign_clusters(X, centroids):
    # START TODO
    raise NotImplementedError("Return nearest-centroid index for every row")


**只检查刚才这个函数：** 填完并运行函数定义后，把下格 `TRY_ASSIGN_CLUSTERS` 改为 `True`。
先用这个小例子核对输入输出，再继续下一步。

In [ ]:
TRY_ASSIGN_CLUSTERS = False
if TRY_ASSIGN_CLUSTERS:
    assert np.array_equal(assign_clusters(np.array([[0.], [9.]]), np.array([[1.], [8.]])), [0, 1])
    print("assign_clusters: example passed")


### 2. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 3. update_centroids

**要写什么：** `update_centroids(X,assignment,k,old_centroids)` 根据分组重算中心。
对每个簇内的样本逐列取平均，返回 `(k,d)`。例如一簇中两个点为 `[0,2]` 与 `[2,4]`，
新中心是 `[1,3]`。如果某簇没有点，保留它的旧中心，避免空数组均值产生 NaN。

In [ ]:
def update_centroids(X, assignment, k, old_centroids):
    # START TODO
    raise NotImplementedError("Return one mean vector per cluster; handle empty clusters")


**只检查刚才这个函数：** 填完并运行函数定义后，把下格 `TRY_UPDATE_CENTROIDS` 改为 `True`。
先用这个小例子核对输入输出，再继续下一步。

In [ ]:
TRY_UPDATE_CENTROIDS = False
if TRY_UPDATE_CENTROIDS:
    example = update_centroids(np.array([[0., 2.], [2., 4.]]), np.array([0, 0]), 1, np.array([[0., 0.]]))
    assert np.allclose(example, [[1., 3.]])
    print("update_centroids: example passed")


### 4. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 5. fit_kmeans

**要写什么：** `fit_kmeans(X,k,rng,max_steps=50)` 将刚才两步循环起来。
先用传入的 `rng` 从 `X` 不放回抽取 `k` 行作为中心，分配样本并记录初始损失。
每轮更新中心、重新分配、再记一次损失，最多运行 `max_steps` 轮。

$$J=\frac1m\sum_i\lVert X_i-c_{a_i}\rVert^2.$$

`a_i` 是样本的簇编号，`c` 是中心。返回 `(centroids,assignment,history)`，
shape 依次是 `(k,d)`、`(m,)`、一维数组。最终编号应对应最终中心。

In [ ]:
def fit_kmeans(X, k, rng, max_steps=50):
    # START TODO
    raise NotImplementedError("Initialize and alternate assignment/update")


### 6. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 7. 串起来：完整实验

image 展平为 pixels:(2304,3) → K-means 找 4 个颜色中心 → 用各自中心替换像素 → reshape 成原图大小。这里只减少颜色种类，不代表 NPY 文件一定更小。

完成相关函数后，打开下格的 `RUN_P1_CHECKS` 并运行。

In [ ]:
RUN_P1_CHECKS = False
if RUN_P1_CHECKS:
    pixels = image.reshape(-1, 3)
    centroids, assignment, history = fit_kmeans(pixels, 4, np.random.default_rng(229))
    compressed = centroids[assignment].reshape(image.shape)
    print("loss history:", history)
    assert len(history) >= 2
    assert np.all(np.isfinite(history))
    assert compressed.shape == image.shape
    assert np.all(np.diff(history) <= 1e-10)
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    axes[0].imshow(image); axes[0].set_title("original")
    axes[1].imshow(np.clip(compressed, 0, 1)); axes[1].set_title("4 colors")
    for ax in axes: ax.axis("off")
    plt.show()
else:
    print("P1 checks are off.")


> **你的答案（请双击本格后填写）**
>
> TODO：解释为什么每次 assignment/update 都不会增加 K-means objective。

## Problem 2 — PCA 降到一维再重建

**本题目的：** 把二维点压缩成一个数，再从这个数重建二维位置。观察保留一个主要方向后，哪些信息丢失了。

### 1. fit_pca

**要写什么：** `fit_pca(X,n_components=1)` 找出数据变化最大的方向。
输入 `X:(m,d)`，返回逐列均值 `mean:(d,)` 与主方向 `components:(d,k)`，其中 `k=n_components`。
先减均值，本题按 $C=X_c^TX_c/m$ 定义协方差，用 `np.linalg.eigh` 求特征值和特征向量，
按特征值从大到小取前 k 列。本题是 `(180,2)` → 均值 `(2,)` 与方向 `(2,1)`。
特征向量的正负都可以，不要把符号不同误判成错误。

In [ ]:
def fit_pca(X, n_components=1):
    # Return mean and principal directions sorted by decreasing eigenvalue.
    # START TODO
    raise NotImplementedError("Center, form covariance, and use np.linalg.eigh")


### 2. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 3. pca_transform

**要写什么：** `pca_transform(X,mean,components)` 将数据投影到刚才学到的方向。
使用训练时的均值中心化后，再投影。返回 `Z:(m,k)`；本题每个二维点变为一个坐标，得到 `(180,1)`。
此函数不重新拟合方向。请在注释里解释保留了哪个方向的信息。

In [ ]:
def pca_transform(X, mean, components):
    # START TODO
    raise NotImplementedError("Project centered data")


### 4. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 5. pca_inverse_transform

**要写什么：** `pca_inverse_transform(Z,mean,components)` 把低维坐标放回原空间，
再加回均值。输入 `Z:(m,k)`，返回 `(m,d)`；本题恢复为 `(180,2)`。
丢掉的方向无法恢复，因此“重建”通常不等于原始数据。

In [ ]:
def pca_inverse_transform(Z, mean, components):
    # START TODO
    raise NotImplementedError("Reconstruct in original space")


### 6. 准备数据

先运行这格准备本题数据，后面的函数将使用它。

In [ ]:
# END TODO


### 7. 串起来：完整实验

拟合均值和主方向 → 投影 Z → 重建二维点 → 比较重建 MSE 和全部预测成均值的基线 → 画原点与重建点。

完成相关函数后，打开下格的 `RUN_P2_CHECKS` 并运行。

In [ ]:
RUN_P2_CHECKS = False
if RUN_P2_CHECKS:
    mean, components = fit_pca(points, n_components=1)
    Z = pca_transform(points, mean, components)
    reconstructed = pca_inverse_transform(Z, mean, components)
    error = np.mean((points - reconstructed) ** 2)
    baseline = np.mean((points - mean) ** 2)
    print("components:", components.shape, "Z:", Z.shape)
    print("reconstruction MSE:", error, "baseline:", baseline)
    assert components.shape == (2, 1)
    assert Z.shape == (len(points), 1)
    assert error < baseline
    plt.figure(figsize=(6, 5))
    plt.scatter(points[:, 0], points[:, 1], s=12, alpha=0.5, label="original")
    plt.scatter(reconstructed[:, 0], reconstructed[:, 1], s=12, label="reconstructed")
    plt.xlabel("coordinate 1")
    plt.ylabel("coordinate 2")
    plt.axis("equal")
    plt.legend()
    plt.title("PCA: 2D to 1D and back")
    plt.show()
else:
    print("P2 checks are off.")


## Problem 3 — 一维两成分 GMM 的 EM（选做）

**本题目的：** 选做：假设一批数来自两个高斯分布，在不知道各样本来源的情况下估计两个分布。先估计归属概率，再用概率更新参数，交替执行。

### 1. 准备数据

先运行这格准备本题数据，后面的函数将使用它。`x_gmm` 是 200 个一维观测，学习时不提供它们属于哪一组。

In [ ]:
gmm_rng = np.random.default_rng(229)
x_gmm = np.concatenate([
    gmm_rng.normal(-2.0, 0.5, size=80),
    gmm_rng.normal(2.0, 0.8, size=120),
])


### 2. gmm_e_step

**要写什么：** `gmm_e_step(x,weights,means,variances)` 估计每个观测来自两个分布的概率。
`x:(m,)`；另外三个数组均为 `(2,)`，分别是比例、均值、方差（不是标准差）。

$$f_k(x_i)=\frac{\exp(-(x_i-\mu_k)^2/(2\sigma_k^2))}{\sqrt{2\pi\sigma_k^2}},
\qquad r_{ik}=\frac{\pi_k f_k(x_i)}{\sum_j\pi_j f_j(x_i)}.$$

返回 `responsibilities:(m,2)`；例如一行 `[0.3,0.7]` 表示两个归属概率，行和必须为 1。
若归一化分母为 0，要检查实现并报错，不要任意修改概率后继续。

In [ ]:
def gmm_e_step(x, weights, means, variances):
    # START TODO
    raise NotImplementedError("Return responsibilities with shape (m, k)")


### 3. 准备数据

先运行这格准备本题数据，后面的函数将使用它。`x_gmm` 是 200 个一维观测，学习时不提供它们属于哪一组。

In [ ]:
# END TODO


### 4. gmm_m_step

**要写什么：** `gmm_m_step(x,responsibilities)` 用刚才的归属概率更新分布参数。
令 $N_k=\sum_i r_{ik}$。新比例是 $N_k/m$，新均值是以 $r_{ik}$ 加权的 `x` 平均值，
新方差是相对**本轮新均值**的加权平方偏差平均值；后两者的分母都是 $N_k$。
方差设下限 `1e-8`；若 $N_k=0$ 则检查实现并报错。
返回 `(weights,means,variances)`，每个数组为 `(2,)`。比例和为 1，方差为正。

In [ ]:
def gmm_m_step(x, responsibilities):
    # START TODO
    raise NotImplementedError("Return updated weights, means, and variances")


### 5. 准备数据

先运行这格准备本题数据，后面的函数将使用它。`x_gmm` 是 200 个一维观测，学习时不提供它们属于哪一组。

In [ ]:
# END TODO


### 6. 串起来：完整实验

初始化两个分布 → E-step 求概率 → M-step 更新参数，重复 30 轮。监测 helper 只用于记录 log-likelihood，数值容差内应不下降；E/M 更新仍由你实现。

完成相关函数后，打开下格的 `RUN_P3_CHECKS` 并运行。

In [ ]:
RUN_P3_CHECKS = False
if RUN_P3_CHECKS:
    def gmm_log_likelihood(x, weights, means, variances):
        # Stable monitoring helper; the E/M updates remain your task.
        log_density = -0.5 * (
            np.log(2 * np.pi * variances)[None, :]
            + (x[:, None] - means[None, :]) ** 2 / variances[None, :]
        )
        return np.logaddexp.reduce(np.log(weights)[None, :] + log_density, axis=1).sum()

    weights = np.array([0.5, 0.5])
    means = np.array([-1.0, 1.0])
    variances = np.array([1.0, 1.0])
    history = [gmm_log_likelihood(x_gmm, weights, means, variances)]
    for _ in range(30):
        responsibilities = gmm_e_step(x_gmm, weights, means, variances)
        assert responsibilities.shape == (len(x_gmm), 2)
        assert np.all(np.isfinite(responsibilities))
        assert np.all(responsibilities >= 0)
        assert np.allclose(responsibilities.sum(axis=1), 1.0)
        weights, means, variances = gmm_m_step(x_gmm, responsibilities)
        assert all(a.shape == (2,) for a in (weights, means, variances))
        assert all(np.all(np.isfinite(a)) for a in (weights, means, variances))
        assert np.all(weights > 0) and np.isclose(weights.sum(), 1.0)
        assert np.all(variances > 0)
        history.append(gmm_log_likelihood(x_gmm, weights, means, variances))
    assert np.all(np.diff(history) >= -1e-8)
    print("log-likelihood:", history[0], "->", history[-1])
    print("weights:", weights)
    print("means:", means)
    print("variances:", variances)
else:
    print("P3 checks are off. This problem is optional.")


## Checks

- [ ] K-means objective 不随迭代增加。
- [ ] PCA 的 components、投影和重建 shape 全部写对。
- [ ] 我能解释 K-means hard assignment 与 GMM soft responsibility 的差别。

## Next Steps

Spring 2026 的 diffusion、representation、LLM 和 RL 暂不伪装成官方 problem set。
后续只在找到可靠课堂案例或明确标注“原创 lecture lab”时再添加。